In [ ]:
%sql
--- Step 1: Initialization
CREATE CATALOG IF NOT EXISTS demo_catalog;
CREATE SCHEMA IF NOT EXISTS demo_catalog.demo_schema;
USE CATALOG demo_catalog;
USE SCHEMA demo_schema;

In [ ]:
%sql 
--- Step 2: Create a table **customers** with a column "email". This column "email" is a sensitive column. 
CREATE OR REPLACE TABLE demo_catalog.demo_schema.customers (
  customer_id INT,
  name STRING,
  email STRING,
  ssn STRING
);

In [ ]:
%sql 
--- Step 3:  
INSERT INTO demo_catalog.demo_schema.customers VALUES
  (1, 'Richard Feynman', 'frichard@aol.com', '123-45-6789'),
  (2, 'Robert Oppenheimer', 'orobert@aol.com','987-65-4321'),
  (3, 'Albert Einstein',  'ealbert@yahoo.com', '585-95-1056');


In [ ]:
%sql 
SELECT * FROM demo_catalog.demo_schema.customers;

customer_id,name,email,ssn
1,Richard Feynman,frichard@aol.com,123-45-6789
2,Robert Oppenheimer,orobert@aol.com,987-65-4321
3,Albert Einstein,ealbert@yahoo.com,585-95-1056


In [ ]:
%sql
-- Step 4: Create the masking function
-- This checks the querying user's email. Replace with YOUR actual login email
-- to prove the "unmasked" branch, then swap in a fake one to see it mask.
CREATE OR REPLACE FUNCTION demo_catalog.demo_schema.mask_ssn(ssn STRING, email STRING)
RETURNS STRING
RETURN CASE
  WHEN email LIKE '%@aol.com' THEN ssn
  ELSE '***-**-****'
END;

In [ ]:
%sql 
ALTER TABLE demo_catalog.demo_schema.customers ALTER COLUMN ssn 
SET MASK demo_catalog.demo_schema.mask_ssn USING COLUMNS(email); -- we have used two columns ssn and email. 
                                                                 -- for extra parameter 'email' in function 'mask_ssn'
                                                                 -- we have to use USING COLUMNS clause. 
SELECT * FROM demo_catalog.demo_schema.customers;
     
-- Step 5: Clean up
-- DROP TABLE demo_catalog.demo_schema.customers;
-- DROP SCHEMA IF EXISTS demo_catalog.demo_schema;
-- DROP CATALOG IF EXISTS demo_catalog;

customer_id,name,email,ssn
1,Richard Feynman,frichard@aol.com,123-45-6789
2,Robert Oppenheimer,orobert@aol.com,987-65-4321
3,Albert Einstein,ealbert@yahoo.com,***-**-****
